### Criando funcao para diferenciar componentes de equacao: coeficientes e variaveis

In [273]:
# Definindo funcao que separa coeficientes e variaveis em listas
def funcao_coef_variaveis(string_equacao):
    # isto esta ok, e consiste em tirar os sinais positivos ou negativos dos coeficientes
    lista_sinal_coeficientes = []
    for elemento in string_equacao:
        if elemento == '+':
            lista_sinal_coeficientes.append(elemento)
        if elemento == '-':
            lista_sinal_coeficientes.append(elemento)
    
    # isto esta ok, e consiste em tirar os coeficientes e que a mesma dimensao da lista dos sinais concida com os coeficientes
    separador_tipo_variavel = []
    novo_string = string_equacao
    for i in range(1,100):
        for sep in ['x_', 's_', 'e_']:
            novo_string = novo_string.replace(f'{sep}{i}', 'o')
    novo_string = novo_string.replace(' ', '').replace('+', '').replace('-', '')    
    lista_coeficiente = novo_string.split('o')    
    if '' in lista_coeficiente:
        lista_coeficiente.remove('')
    elif ' ' in lista_coeficiente:
        lista_coeficiente.remove(' ')        
    
    # aqui separa pelos segmentos que contem coeficientes e x_ ou s_ ou e_
    separando_equacao = string_equacao.replace('-', '+').split('+')
    if '' in separando_equacao:
        separando_equacao.remove('')
    elif ' ' in separando_equacao:
        separando_equacao.remove(' ')
    dicionario_c={}

    # se utiliza a separacao em partes para criar um dicionario que outorgue coeficientes as respectivas variaveis 
    for i, elemento in enumerate(separando_equacao):
        if 'x' in elemento:
            coeficiente, variavel = elemento.split('x')
            dicionario_c[f'x{variavel.strip()}'] = float(f'{lista_sinal_coeficientes[i]}{coeficiente.strip()}')
        elif 's' in elemento:
            coeficiente, variavel = elemento.split('s')
            dicionario_c[f's{variavel.strip()}'] = float(f'{lista_sinal_coeficientes[i]}{coeficiente.strip()}')
        elif 'e' in elemento:
            coeficiente, variavel = elemento.split('e')
            dicionario_c[f'e{variavel.strip()}'] = float(f'{lista_sinal_coeficientes[i]}{coeficiente.strip()}')
        elif 'a' in elemento:
            coeficiente, variavel = elemento.split('a')
            dicionario_c[f'a{variavel.strip()}'] = float(f'{lista_sinal_coeficientes[i]}{coeficiente.strip()}')
        
    return dicionario_c

### Funcao para criar matriz de restricoes A, considerando variaveis folga, excedentes, e artificiais

In [274]:
def matriz_restricoes(lista_restricoes):
        # caso <=, entao adicionar s_i (variavel de folga)
    i = 1
    tupla_matriz_A = ()
    tupla_coef_b = ()
    for restricao in lista_restricoes:
        if '<=' in restricao:
            equacao_restricao, coef_b = restricao.split('<=') 
            equacao_restricao = equacao_restricao + f'+ 1s_{i}'
        
        # caso >=, entao adicionar e_i (variavel excedente) e tambem a_i (variavel artificial)
        elif '>=' in restricao:
            equacao_restricao, coef_b = restricao.split('>=')
            equacao_restricao = equacao_restricao + f'- 1e_{i} '
            i += 1
            equacao_restricao = equacao_restricao + f'+ 1a_{i} '
        
        # caso ==, entao adicionar a_i
        elif '==' in restricao:
            equacao_restricao, coef_b = restricao.split('==')
            equacao_restricao = equacao_restricao + f'+ 1a_{i} '

        dicionario_restricao = funcao_coef_variaveis(equacao_restricao)  
        tupla_matriz_A += (dicionario_restricao, )
        tupla_coef_b += (float(coef_b.strip()), )
        
        i += 1
    
    return tupla_matriz_A, tupla_coef_b


### Funcao conversao dicionario de restricao em lista de restricoes

In [275]:
def funcao_conversao_dicionario_restricao_em_lista_restricao(tupla_restricoes, lista_todas_variaveis):
    lista_matriz_A = []
    lista_restricao = []
    for dicionario_restricao in tupla_restricoes:
        lista_restricao = []
        for variavel in lista_todas_variaveis:
            if variavel in dicionario_restricao:
                lista_restricao.append(dicionario_restricao[variavel])
            else:   
                lista_restricao.append(0)
        lista_matriz_A.append(lista_restricao)
    return lista_matriz_A

### Funcao conversao dicionario de funcao em lista de funcao

In [276]:
def funcao_conversao_dicionario_funcaoobjetivo_em_lista_funcaoobjetivo(dicionario_fo, lista_todas_variaveis):
    lista_fo = []
    for variavel in lista_todas_variaveis:
        
        if variavel in dicionario_fo:
            lista_fo.append(dicionario_fo[variavel])
        else:   
            lista_fo.append(0)
    return lista_fo

# lista_fo = funcao_conversao_dicionario_funcaoobjetivo_em_lista_funcaoobjetivo(dicionario_fo, lista_todas_variaveis)


## **Tratamento para obter os numeros dos vetores e matrizes**

In [277]:
# Definir a f.o.: min. ou max. e digite a funcao
fo = 'max. + 6x_1 - 7x_2 - 4x_3'
fo_min_max, fo_equacao = fo.split('.')
fo_equacao.replace(' ', '')

# Obter variaveis e coeficientes separados numa funcao
dicionario_fo =  funcao_coef_variaveis(fo_equacao)

# Definir as restricoes, tornar os coeficientes a direita positivos
g_1 = '+ 2x_1 + 5x_2 - 1x_3 <= 18'
g_2 = '- 1x_1 + 1x_2 + 2x_3 >= 14'
h_3= '+ 3x_1 + 2x_2 + 2x_3 == 26 '
lista_restricoes = [g_1, g_2, h_3]
        
# Procesamento para obtencao da matriz em forma de tupla(dicionarios)
tupla_restricoes, tupla_coef_b = matriz_restricoes(lista_restricoes)

# Organizando tudo em vetor
    # Tirar todas as chaves da tupla e organizar em order
lista_todas_variaveis = []
for dicionario in tupla_restricoes:
    lista_chaves = list(dicionario.keys())
    for chave in lista_chaves:
        if chave not in lista_todas_variaveis:
            lista_todas_variaveis.append(chave)
    
# Organizar cada restricao em forma de vetor, adicionar os 0s caso nao haja coeficiente
lista_matriz_A = funcao_conversao_dicionario_restricao_em_lista_restricao(tupla_restricoes, lista_todas_variaveis)

# Printar funcao objetivo, restricoes, e vetor b    
lista_coef_b = list(tupla_coef_b)
lista_fo = funcao_conversao_dicionario_funcaoobjetivo_em_lista_funcaoobjetivo(dicionario_fo, lista_todas_variaveis)

print('variaveis:', lista_todas_variaveis)
print('vetor fo:', lista_fo)
print('matriz A:' , lista_matriz_A)
print('vetor b:', lista_coef_b)


variaveis: ['x_1', 'x_2', 'x_3', 's_1', 'e_2', 'a_3', 'a_4']
vetor fo: [6.0, -7.0, -4.0, 0, 0, 0, 0]
matriz A: [[2.0, 5.0, -1.0, 1.0, 0, 0, 0], [-1.0, 1.0, 2.0, 0, -1.0, 1.0, 0], [3.0, 2.0, 2.0, 0, 0, 0, 1.0]]
vetor b: [18.0, 14.0, 26.0]


## **Definindo o tableau para minimizar a funcao objetivo**
1. Definir base do tableau: Escolher as bases baseado nas folgas com indice nao negativo (variaveis >=0)
    - Utilizar a lista de variaveis (x, s, e, a) e fazer uma leitura de cada fila de restricao.
    - Caso a variavel seja a, e, ou s, e o coeficiente respectivo for positivo, entao adicionar essa variavel como base.
2. Calcular $C_j - Z_j$ utilizando:
    - na Fase 1, a maximizacao da funcao $-W = \sum_{i=1}^{n} -A_i$
    - na Fase 2, a maximizacao da funcao objeito original

In [278]:
# Caso haja variaveis artificias
variaveis_artificiais = []
teste_a = 'a_'
for i in range(1, 20):
    variacao_teste_a = f'{teste_a}{i}'
    if variacao_teste_a in lista_todas_variaveis:
        variaveis_artificiais.append(variacao_teste_a)
if len(variaveis_artificiais) > 0:
    print('Existem variaveis artificiais:\n', variaveis_artificiais) 
    M = 10

# Defino cada vetor do tableau
print('\nvariaveis:', lista_todas_variaveis)
print('vetor fo:', lista_fo)
print('matriz A:' , lista_matriz_A)
print('vetor b:', lista_coef_b)

Existem variaveis artificiais:
 ['a_3', 'a_4']

variaveis: ['x_1', 'x_2', 'x_3', 's_1', 'e_2', 'a_3', 'a_4']
vetor fo: [6.0, -7.0, -4.0, 0, 0, 0, 0]
matriz A: [[2.0, 5.0, -1.0, 1.0, 0, 0, 0], [-1.0, 1.0, 2.0, 0, -1.0, 1.0, 0], [3.0, 2.0, 2.0, 0, 0, 0, 1.0]]
vetor b: [18.0, 14.0, 26.0]


In [279]:
# Funcao para calculo de Z_j e Z_j-C_j
def funcao_calculo_Z_j_e_Z_j_menos_C_j(lista_todas_variaveis, coeficiente_base_tableau_C_j, vetor_coeficientes_C_B):
    vetor_Z_j = []
    for i, variavel in enumerate(lista_todas_variaveis):
        Z = 0
        for k, elemento in enumerate(vetor_coeficientes_C_B):
            valor_A_j = lista_matriz_A[k][i] 
            Z += elemento*valor_A_j
        vetor_Z_j.append(Z)
        
    # 5. Calcula-se o valor C_j-Z_j
    vetor_C_j_menos_Z_j = []
    for i, variavel in enumerate(lista_todas_variaveis):
        C_j_menos_Z_j = 0    
        C_j_menos_Z_j = coeficiente_base_tableau_C_j[i]-vetor_Z_j[i]
        vetor_C_j_menos_Z_j.append(C_j_menos_Z_j)
        
    return vetor_Z_j, vetor_C_j_menos_Z_j


### Aqui vou fazer a funcao que vai fazer as iteracoes para criar uma nova matriz_A e um novo vetor_b
- Destaca-se que embaixo se encontra a resolucao inicial desta funcao

In [280]:
def funcao_calculo_iteracoes(lista_todas_variaveis, coeficiente_base_tableau_C_j, lista_matriz_A, vetor_variaveis_C_B, vetor_coeficientes_C_B, lista_coef_b, coluna_pivo):
    indice_coluna_pivo = lista_todas_variaveis.index(coluna_pivo) # lembrando que eh o valor da coluna que substitui o valor da fila
    indice_fila_pivo = vetor_variaveis_C_B.index(coluna_pivo)
    valor_pivo = lista_matriz_A[indice_fila_pivo][indice_coluna_pivo]
    
    for j, elemento_fila in enumerate(lista_matriz_A[indice_fila_pivo]):
        valor_pivo_substituir = float(elemento_fila / valor_pivo)
        lista_matriz_A[indice_fila_pivo][j] = valor_pivo_substituir
    lista_coef_b[indice_fila_pivo] = lista_coef_b[indice_fila_pivo] / valor_pivo

    for i, fila_matriz in enumerate(lista_matriz_A):        
        # Tratando matriz A
        if i != indice_fila_pivo:
            valor_a_eliminar = fila_matriz[indice_coluna_pivo]
            for j, elemento_fila in enumerate(fila_matriz):
                valor_a_substituir_em_fila = (lista_matriz_A[indice_fila_pivo][j] * (-valor_a_eliminar) ) + lista_matriz_A[i][j]
                lista_matriz_A[i][j] = valor_a_substituir_em_fila
        # Tratando vetor b
            valor_a_substituir_em_vetor_b = lista_coef_b[indice_fila_pivo] * (-valor_a_eliminar) + lista_coef_b[i] 
            lista_coef_b[i] = valor_a_substituir_em_vetor_b
    
    return lista_matriz_A, lista_coef_b, vetor_coeficientes_C_B

## **Começam as iterações no tableau para minimizar a funcao objetivo**

In [281]:
# # 1. Utiliza-se o valor pivo para calcular os novos valores da fila pivo
# indice_coluna_pivo = lista_todas_variaveis.index(coluna_pivo) # lembrando que eh o valor da coluna que substitui o valor da fila
# indice_fila_pivo = vetor_variaveis_C_B.index(coluna_pivo)
# valor_pivo = lista_matriz_A[indice_coluna_pivo][indice_fila_pivo]

#     # Aqui atualizo a fila pivo com os valores dividos para o valor pivo, assim o valor pivo deve ficar=1, para matriz A e vetor b
# print('matriz A incio:', lista_matriz_A)
# print('vetor_b sem tratar:', lista_coef_b)
# for j, elemento_fila in enumerate(lista_matriz_A[indice_fila_pivo]):
#     valor_pivo_substituir = float(elemento_fila / valor_pivo)
#     lista_matriz_A[indice_fila_pivo][j] = valor_pivo_substituir
# lista_coef_b[indice_fila_pivo] = lista_coef_b[indice_fila_pivo] / valor_pivo
# print('Tratada fila pivo:', lista_matriz_A)

# # 2. Compara-se o valor pivo com os outros valores da mesma coluna, mas diferente fila,
# # e se armazena numa nova variavel para , dentro desse loop, fazer calculos
# # Atualiza-se a nova matriz_A, vetor_b, C_j-Z_j, etc, para serem analisados no loop do while
# print('vetor_b so tratado fila pivo:', lista_coef_b)
# for i, fila_matriz in enumerate(lista_matriz_A):    
    
#     # Tratando matriz A
#     if i != indice_fila_pivo:
#         valor_a_eliminar = fila_matriz[indice_coluna_pivo]
#         for j, elemento_fila in enumerate(fila_matriz):
#             valor_a_substituir_em_fila = (lista_matriz_A[indice_fila_pivo][j] * (-valor_a_eliminar) ) + lista_matriz_A[i][j]
#             lista_matriz_A[i][j] = valor_a_substituir_em_fila

#     # Tratando vetor b
#         valor_a_substituir_em_vetor_b = lista_coef_b[indice_fila_pivo] * (-valor_a_eliminar) + lista_coef_b[i] 
#         lista_coef_b[i] = valor_a_substituir_em_vetor_b

#     # Atualizando vetor C_B
# for i, variavel_C_B in enumerate(vetor_variaveis_C_B):
#     # if variavel_C_B in lista_todas_variaveis:
#     indice_a_ser_considerado = lista_todas_variaveis.index(variavel_C_B)
#     valor_a_ser_substituido_em_C_B = coeficiente_base_tableau_C_j[indice_a_ser_considerado]       
#     vetor_coeficientes_C_B[i] = valor_a_ser_substituido_em_C_B

# print('matriz A final:', lista_matriz_A)
# print('vetor_b tratado', lista_coef_b)
# print('vetor variaveis C_B', vetor_variaveis_C_B)
# print('vetor coeficientes C_B',vetor_coeficientes_C_B)


# vetor_Z_j, vetor_C_j_menos_Z_j = funcao_calculo_Z_j_e_Z_j_menos_C_j(lista_todas_variaveis, coeficiente_base_tableau_C_j, vetor_coeficientes_C_B)
# print(lista_todas_variaveis)
# print(coeficiente_base_tableau_C_j)
# print('vetor_Z_j', vetor_Z_j) 
# print('vetor_C_j_menos_Z_j', vetor_C_j_menos_Z_j)


In [282]:
# 1 Define-se os valores de todas as variaveis base do tableau C_j (consideram-se todas as variaveis) 
coeficiente_base_tableau_C_j = []
for i, variavel in enumerate(lista_todas_variaveis):
    if 'x' in variavel:
        coeficiente_base_tableau_C_j.append(lista_fo[i])
    elif 'a' in variavel:
        coeficiente_base_tableau_C_j.append(-M)
    else:
        coeficiente_base_tableau_C_j.append(0)
print('coeficiente_base_tableau_C_j: ', coeficiente_base_tableau_C_j)

# 2. Cria-se vetor C_B que contenha as variaveis s_i, e a_i
vetor_variaveis_C_B = []  
for variavel in lista_todas_variaveis:
    if 's' in variavel:
        vetor_variaveis_C_B.append(variavel)
    elif 'a' in variavel:
        vetor_variaveis_C_B.append(variavel)
print('vetor_variaveis_C_B: ', vetor_variaveis_C_B)

# 3 Define-se o vetor valores C_B baseado nos valores de s_i, e a_i
vetor_coeficientes_C_B = []
for i, variavel in enumerate(lista_todas_variaveis):
    if variavel in vetor_variaveis_C_B:
        vetor_coeficientes_C_B.append(coeficiente_base_tableau_C_j[i])
print('vetor_coeficientes_C_B: ', vetor_coeficientes_C_B)


## -------------------------------------------------------------------------------------------------------------------------------------------------------
## -------------------------------------------------------------------------------------------------------------------------------------------------------
## AQUI DEVE COMECAR A FUNCAO PARA Z_j e C_j-Z_j

vetor_Z_j, vetor_C_j_menos_Z_j = funcao_calculo_Z_j_e_Z_j_menos_C_j(lista_todas_variaveis, coeficiente_base_tableau_C_j, vetor_coeficientes_C_B)

# # 4. Calcula-se o valor de Z_j
# vetor_Z_j = []
# for i, variavel in enumerate(lista_todas_variaveis):
#     Z = 0
#     for k, elemento in enumerate(vetor_coeficientes_C_B):
#         valor_A_j = lista_matriz_A[k][i] 
#         Z += elemento*valor_A_j
#     vetor_Z_j.append(Z)
# print('vetor Z_j: ', vetor_Z_j)
    
# # 5. Calcula-se o valor C_j-Z_j
# vetor_C_j_menos_Z_j = []
# for i, variavel in enumerate(lista_todas_variaveis):
#     C_j_menos_Z_j = 0    
#     C_j_menos_Z_j = coeficiente_base_tableau_C_j[i]-vetor_Z_j[i]
#     vetor_C_j_menos_Z_j.append(C_j_menos_Z_j)
# print('vetor_C_j_menos_Z_j: ',vetor_C_j_menos_Z_j)

## -------------------------------------------------------------------------------------------------------------------------------------------------------
## -------------------------------------------------------------------------------------------------------------------------------------------------------
## AQUI DEVE COMECAR O LOOP ATE C_j-Z_j <= 0

# 6. Avalia-se se C_j-Z_j <= 0, caso seja, entao finalizou o calculo da Fase 1, mas caso nao seja, 
# se calcula a base que entra-sai ou se faz uma segunda iteracao
while any(e > 0 for e in vetor_C_j_menos_Z_j):
    print('-------------------------------------------------------------------------------------')
    print('# Inicia loop')
    # Caso se calcula a base que entra-sai, entao comeca-se por definir o maior valor de C_j-Z_j, e define-se a coluna a utilizar (.index()) e a variavel
    maior_valor_C_j_Z_j = max(vetor_C_j_menos_Z_j)
    indice_maior_valor_C_j_Z_j = vetor_C_j_menos_Z_j.index(maior_valor_C_j_Z_j) # INDICE COLUNA PIVO
    coluna_pivo = lista_todas_variaveis[indice_maior_valor_C_j_Z_j]

    # Depois, se calcula a razao entre a coluna pivo e o seu correspondente b (vetor_b ou lista_coef_b)
        # Caso o valor da razao sea positivo, eh considerado no vetor razao
    vetor_razao = []
    for i, dividendo in enumerate(lista_coef_b):
        divisor = lista_matriz_A[i][indice_maior_valor_C_j_Z_j]
        razao = dividendo / divisor
        if razao >= 0:
            vetor_razao.append(razao)
        else:
            vetor_razao.append(0)
    
        # O valor minimo do vetor razao vai definir qual elemento sai da variaveis base, e qual elemento entra 
    vetor_razao_sem0 = vetor_razao[:]
    vetor_razao_sem0.remove(0)
    menor_valor_razao = min(vetor_razao_sem0)    
    indice_menor_valor_razao = vetor_razao.index(menor_valor_razao)
    fila_pivo =  vetor_variaveis_C_B[indice_menor_valor_razao]
    
        # Se troca a variavel que sai pela que entra
    print('vetor variaveis C_B:', vetor_variaveis_C_B)
    print('vetor_coeficientes_C_B:', vetor_coeficientes_C_B)    
    vetor_variaveis_C_B[vetor_variaveis_C_B.index(fila_pivo)] = coluna_pivo
        # Atualizando vetor C_B
    for i, variavel_C_B in enumerate(vetor_variaveis_C_B):
        # if variavel_C_B in lista_todas_variaveis:
        indice_a_ser_considerado = lista_todas_variaveis.index(variavel_C_B)
        valor_a_ser_substituido_em_C_B = coeficiente_base_tableau_C_j[indice_a_ser_considerado]       
        vetor_coeficientes_C_B[i] = valor_a_ser_substituido_em_C_B

    print('novo vetor variaveis C_B:', vetor_variaveis_C_B)
    print('novo vetor C_B:', vetor_coeficientes_C_B)
    print('matriz_A:', lista_matriz_A)
    print('vetor b:', lista_coef_b)

    print('\n     # Inicia iteracao')
    lista_matriz_A, lista_coef_b, vetor_coeficientes_C_B = funcao_calculo_iteracoes(
        lista_todas_variaveis, 
        coeficiente_base_tableau_C_j, 
        lista_matriz_A, 
        vetor_variaveis_C_B, 
        vetor_coeficientes_C_B, 
        lista_coef_b,
        coluna_pivo
        )

    print('matriz A tratada:', lista_matriz_A)
    print('vetor_b tratado', lista_coef_b)
    
    print('\n     # Inicia calculo Z_j_e_Z_j_menos_C_j')
    vetor_Z_j, vetor_C_j_menos_Z_j = funcao_calculo_Z_j_e_Z_j_menos_C_j(lista_todas_variaveis, coeficiente_base_tableau_C_j, vetor_coeficientes_C_B)

    print('vetor_Z_j', vetor_Z_j) 
    print('vetor_C_j_menos_Z_j', vetor_C_j_menos_Z_j)

otimo_fo = sum(elemento_C_B*lista_coef_b[i] for i, elemento_C_B in enumerate(vetor_coeficientes_C_B) )

texto_fim_loop = f'''-------------------------------------------------------
-------------------------------------------------------
Fim do loop:
otimo da fo: {-otimo_fo}
valores f.o. fase1: {coeficiente_base_tableau_C_j}
matriz_A: {lista_matriz_A}
vetor_b: {lista_coef_b}'''
print(texto_fim_loop)

            

coeficiente_base_tableau_C_j:  [6.0, -7.0, -4.0, 0, 0, -10, -10]
vetor_variaveis_C_B:  ['s_1', 'a_3', 'a_4']
vetor_coeficientes_C_B:  [0, -10, -10]
-------------------------------------------------------------------------------------
# Inicia loop
vetor variaveis C_B: ['s_1', 'a_3', 'a_4']
vetor_coeficientes_C_B: [0, -10, -10]
novo vetor variaveis C_B: ['s_1', 'x_3', 'a_4']
novo vetor C_B: [0, -4.0, -10]
matriz_A: [[2.0, 5.0, -1.0, 1.0, 0, 0, 0], [-1.0, 1.0, 2.0, 0, -1.0, 1.0, 0], [3.0, 2.0, 2.0, 0, 0, 0, 1.0]]
vetor b: [18.0, 14.0, 26.0]

     # Inicia iteracao
matriz A tratada: [[1.5, 5.5, 0.0, 1.0, -0.5, 0.5, 0.0], [-0.5, 0.5, 1.0, 0.0, -0.5, 0.5, 0.0], [4.0, 1.0, 0.0, 0.0, 1.0, -1.0, 1.0]]
vetor_b tratado [25.0, 7.0, 12.0]

     # Inicia calculo Z_j_e_Z_j_menos_C_j
vetor_Z_j [-38.0, -12.0, -4.0, 0.0, -8.0, 8.0, -10.0]
vetor_C_j_menos_Z_j [44.0, 5.0, 0.0, 0.0, 8.0, -18.0, 0.0]
-------------------------------------------------------------------------------------
# Inicia loop
vetor v